**Homework 6: Polynomial Regression**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

We will continue with the displacement `disp` and `mpg` columns of the `cars` dataset, as in the last assignment. (This time we'll sort these by `disp` to make visualization easier later.)

In [ ]:
cars=pd.read_csv('https://vincentarelbundock.github.io/Rdatasets/csv/causaldata/auto.csv')
disp=np.array(cars.displacement)
mpg=np.array(cars.mpg)

index=np.argsort(disp)
disp=disp[index]
mpg=mpg[index]

In this assignment we will use feature engineering to create polynomial models for predicting `mpg` from `disp`. However, before we start looking at powers of `disp`, we'll need to scale this array to have mean 0 and standard deviation 1 (otherwise, the powers of the entries in `disp` will get too large). To the end, create a `StandardScaler` class that remembers the mean and standard deviation of each column of a feature matrix, and can scale and unscale datasets using these numbers.

In [ ]:
class StandardScaler():
    def __init__(self):
        pass

    def fit(self,X):
        self.mean=X.mean(axis=0)
        self.std=X.std(axis=0)

    def transform(self,X):
        return (X-self.mean)/self.std

    def inverse_transform(self,X):
        return X*self.std+self.mean

Create a `StandardScaler` object, fit it to `disp`, and then create a new array called `scaled_disp`.

In [ ]:
disp_scaler=StandardScaler()
disp_scaler.fit(disp)
scaled_disp=disp_scaler.transform(disp)

In [ ]:
scaled_disp[5]

In [ ]:
disp_scaler.inverse_transform(-1.176317312062453)

In [ ]:
disp[5]

In the previous assignment you built a Linear Regression class, identical to the one packeged with sklearn. We'll import this here:

In [ ]:
from sklearn.linear_model import LinearRegression

To create a higher order polynomial model, you'll have to first create a feature matrix with higher powers of `disp`. Create a class that does this for you for any input array `X`. The `fit_transform` method of this class will return a matrix whose first column is a column of ones (if `include_bias=True`), next column is `X`, next is `X**2`, etc.

In [ ]:
class PolynomialFeatures():
    def __init__(self,degree,include_bias=False):
        self.degree=degree
        self.include_bias=include_bias

    def fit_transform(self,X):
        if self.include_bias:
            out=np.ones((len(X),self.degree+1))
            for i in range(self.degree+1):
              out[:,i]=X**i

        else:
            out=np.zeros((len(X),self.degree))
            for i in range(self.degree):
              out[:,i]=X**(i+1)

        return out

Now, for example, if you wanted to create a matrix whose first column is `[0,1,2,3]` and second column is those values squared, you would do this:

In [ ]:
quad=PolynomialFeatures(2)
quad.fit_transform(np.array([0,1,2,3]))

Generate a matrix whose columns are `scaled_disp` and `scaled_disp**2`.

In [ ]:
scaled_disp2=quad.fit_transform(scaled_disp)

In [ ]:
scaled_disp2[10,1]

In [ ]:
scaled_disp2[20,0]

Create a quadratic model to predict `mpg` from `scaled_disp` by creating a linear model to predict `mpg` from both `scaled_disp` and `scaled_disp**2` (*i.e.* from `scaled_disp2`).

In [ ]:
quadratic_mod=LinearRegression()
quadratic_mod.fit(scaled_disp2,mpg)

Now, apply this model to `scaled_disp2` to create an array of predictions.

In [ ]:
quadratic_mod.predict(np.array([[.5,.5**2]]))[0]

In [ ]:
quad_preds=quadratic_mod.predict(scaled_disp2)

In [ ]:
quad_preds[50]

Now visualize it:

In [ ]:
plt.scatter(disp,mpg)
plt.plot(disp,quad_preds,'-r')
plt.xlabel('Displacement')
plt.ylabel('MPG')

Calculate the RSS of ```quadratic_mod```.

In [ ]:
RSSquad=((quad_preds-mpg)**2).sum()
RSSquad

Now create a cubic model of `mpg` vs `scaled_disp`, visualize it, and calculate its RSS.

In [ ]:
cubic_mod=LinearRegression()
cube=PolynomialFeatures(3)
scaled_disp3=cube.fit_transform(scaled_disp)
cubic_mod.fit(scaled_disp3,mpg)
cubic_preds=cubic_mod.predict(scaled_disp3)

In [ ]:
cubic_preds[50]

In [ ]:
plt.scatter(disp,mpg)
plt.plot(disp,cubic_preds,'-r')
plt.xlabel('Displacement')
plt.ylabel('MPG')

In [ ]:
RSScubic=((cubic_preds-mpg)**2).sum()
RSScubic